# Dell Kit End Date Prediction
**Zakir Patwa · Data Science Case Study · 2026**

**Goal:** predict `kit_end`, the date kitting finishes, for each of the 12,069 orders in `inference.parquet` (kit starts Feb 3 – Mar 7, 2026), using 179,853 historical orders (kit starts Mar 2025 – Jan 2026).

**Deliverables:** this notebook, `README.md` (findings + recommendations), and `Zakir_Patwa_DS_Case_Predictions_2026.npy` (one predicted date per inference row, same order).

**Outline**
1. Problem framing and key decisions
2. Assumptions log (kept up to date as each cleaning/join decision is made)
3. Load data
4. Exploration, organized around three hypotheses: supply constraints, human schedules, warehouse congestion
5. Cleaning
6. Feature engineering
7. Modeling: baselines, classifier vs. regressor, metrics, probability checks, error analysis
8. Inference predictions

## 1. Problem framing and key decisions

Six decisions shape everything downstream. Each one lists the choice, the alternative considered, and the reason.

**1. Target: predict duration in calendar days, not a date.**
`kit_days = kit_end − kit_start`. The model predicts `kit_days`, and the date is rebuilt as `kit_end = kit_start + kit_days`. `kit_start` is known for every inference order, so duration is the only unknown, and duration patterns carry over across months where raw dates never repeat. *Alternative:* business days using the calendar table. Rejected because weekend work is routine (~35k kits start on a Saturday even though the calendar marks Saturday as a weekend), and the calendar only runs through 2026-03-10, barely covering the inference window. Historical median is 1 day, mean 1.34.

**2. Model: build a classifier and a regressor, keep the better one.**
- *Classifier:* LightGBM on five buckets {0, 1, 2, 3, 4+}. Over 94% of orders take 0–3 days (0d 19.5% · 1d 48.4% · 2d 17.0% · 3d 9.1% · 4d+ ~6%), so the buckets cover the practical decision space, and rare 20–76 day outliers can't distort the fit. A "4+" prediction maps to the training median of 4+ orders. It also gives a **probability per bucket**, which powers a high-risk flag for orders likely to run long.
- *Regressor:* LightGBM predicting `kit_days` directly, rounded to whole days.
- *Selection:* whichever scores higher on validation **±1-day accuracy**. LightGBM (gradient-boosted decision trees) is the standard for tabular data: it handles categorical columns and missing values natively and trains in seconds.

**3. Validation: time-based split, not random.**
Train on `kit_start` ≤ 2025-11-30 (146,444 rows) and validate on Dec 2025 – Jan 2026 (33,409 rows, 19%). Inference is entirely in the future (Feb–Mar 2026), so validation has to be too. A random split would let the model learn from Dec 15 to "predict" Dec 14, which inflates scores. Note that validation is *harder* than inference is likely to be: it contains the Dec 23–27, Jan 1 and Jan 15 holidays (0-day kits 14.7% vs 19.5% overall; 4+ 8.4% vs ~6%), and there are no holidays in Feb–Mar. Validation scores are therefore a conservative estimate. After model selection, the final model is retrained on all data through Jan 31.

**4. Baselines: the model must beat three simple rules.**
All medians are computed from training data only.
1. *Always 1 day*: the most common outcome (48%).
2. *Median `kit_days` per `family_desc`*: a rule a planner could run in a spreadsheet.
3. *Median per `family_desc` × `mcid` (warehouse)*: falls back to the family median when a family–warehouse combination has fewer than 30 training orders, and to the global median for families never seen in training.

If the model can't clearly beat baseline 3, its extra complexity isn't justified.

**5. Metrics.**
- **Within ±1 day % (primary)**: close enough for a planner to schedule around.
- **Exact-day %**, **MAE** (average days off), **macro-F1** (does it get the rare buckets right, not just "1 day"?), and a **confusion matrix** (which mistakes it makes).
- **Probability checks** for the risk flag: **log loss** (penalizes confident wrong answers), a **calibration chart** (among orders given a 30% risk, do ~30% actually run long?), and flag **precision/recall** (are the alarms real, and do they catch the slow orders?). The flag is a claim to planners, so it has to be tested, not just produced.

**6. Output format.**
`Zakir_Patwa_DS_Case_Predictions_2026.npy` holds `datetime64[D]` dates (day precision, loads with a plain `np.load`). Before saving, the notebook asserts: length 12,069, no missing dates, no `kit_end` before `kit_start`, and row order identical to `inference.parquet`. A supporting CSV (`Zakir_Patwa_DS_Case_Predictions_2026.csv`: order ID, family, kit start, predicted kit end, predicted days, risk probability) is produced for planners to open in Excel.

## 2. Assumptions log
*Updated as each cleaning, join, or modeling decision is made.*

**Problem setup**
- A1. `kit_start` is known at prediction time for every inference order (it is provided in `inference.parquet`), so predicting duration is legitimate and not leakage.
- A2. Calendar days are the right unit (see Decision 1); weekends are not treated as non-working days.
- A3. Kitting behavior in Feb–Mar 2026 resembles Mar 2025 – Jan 2026 closely enough for historical patterns to apply. No holidays fall in the inference window.

**Reference tables**
- A4. The calendar table is tagged site `FCJ`, but every order is site `KHO`. Assumed to be one factory, so the calendar is joined on date only.
- A5. Calendar `day_of_week` is encoded 1 = Saturday, 2 = Sunday (per the data dictionary).

**Modeling**
- A6. A "4+ days" prediction is converted to a date using the training median of 4+ orders.
- A7. Families never seen in training (10 in inference) get the global median in the baselines. The model handles them through its other features.
- A8. All medians, encodings and fallbacks are fit on training rows only, so validation stays honest.

## 3. Load data

Five tables, all read from `Data/` (not committed to the repo):

| Table | Grain | Role |
|---|---|---|
| `train` | one row per order line, with `kit_end` | learn from |
| `inference` | same columns, minus `kit_end` | predict; **row order must be preserved** |
| `expected_minutes` | one row per product family | engineered labor-time estimate per unit |
| `calendar` | one row per date | weekends, holidays, end-of-quarter flags |
| `capacity` | one row per date (after dedupe) | overtime hours |

The shape checks below catch a wrong or truncated file before any analysis starts.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)

DATA_DIR = Path("Data")

train = pd.read_parquet(DATA_DIR / "train.parquet")
inference = pd.read_parquet(DATA_DIR / "inference.parquet")
expected_minutes = pd.read_parquet(DATA_DIR / "expected_minutes.parquet")
calendar = pd.read_parquet(DATA_DIR / "calendar.parquet")
capacity = pd.read_parquet(DATA_DIR / "capacity.parquet")

tables = {
    "train": train,
    "inference": inference,
    "expected_minutes": expected_minutes,
    "calendar": calendar,
    "capacity": capacity,
}
for name, df in tables.items():
    print(f"{name:<17} {df.shape[0]:>8,} rows x {df.shape[1]:>2} cols")

# Guard against a wrong or truncated file
assert train.shape == (179_853, 19)
assert inference.shape == (12_069, 18)

# Inference has every train column except the target
print("\nIn train but not inference:", sorted(set(train.columns) - set(inference.columns)))

train              179,853 rows x 19 cols
inference           12,069 rows x 18 cols
expected_minutes       260 rows x  4 cols
calendar               370 rows x 17 cols
capacity           191,934 rows x  4 cols

In train but not inference: ['kit_end']
